In [ ]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    f1_score,
    confusion_matrix,
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    auc
)

from imblearn.over_sampling import SMOTE

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.initializers import GlorotUniform
import tensorflow as tf

import matplotlib.pyplot as plt
import seaborn as sns

# =========================
# Configuration
# =========================
DATASET_DIR = "datasets/"
N_SPLITS = 5
EPOCHS = 100
BATCH_SIZE = 32
RANDOM_STATE = 42

model_configs = {
    "simple": [128, 64],
    "complex": [256, 128, 64]
}

dataset_files = [f for f in os.listdir(DATASET_DIR) if f.endswith(".csv")]

results = []
confusion_matrices = []
roc_curves = []
pr_curves = []

# =========================
# Main Loop
# =========================
for file in dataset_files:
    df = pd.read_csv(os.path.join(DATASET_DIR, file))
    df = df.drop(columns=['sample'], errors='ignore').dropna()

    X = df.drop(columns=['stage']).values
    y = df['stage'].values

    # -------------------------
    # Encode labels (BINARY)
    # -------------------------
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    labels = le.classes_

    assert len(np.unique(y_encoded)) == 2, "Binary classification only."

    # -------------------------
    # Hold-out Test Set (10%)
    # -------------------------
    X_trainval, X_test, y_trainval, y_test = train_test_split(
        X,
        y_encoded,
        test_size=0.1,
        stratify=y_encoded,
        random_state=RANDOM_STATE
    )

    skf = StratifiedKFold(
        n_splits=N_SPLITS,
        shuffle=True,
        random_state=RANDOM_STATE
    )

    for config_name, layers in model_configs.items():
        fold_acc = []
        fold_f1 = []

        # =========================
        # Cross-Validation
        # =========================
        for train_idx, val_idx in skf.split(X_trainval, y_trainval):
            X_train_raw = X_trainval[train_idx]
            X_val_raw = X_trainval[val_idx]
            y_train = y_trainval[train_idx]
            y_val = y_trainval[val_idx]

            # -------------------------
            # Scaling (TRAIN ONLY)
            # -------------------------
            scaler = StandardScaler()
            X_train_scaled = scaler.fit_transform(X_train_raw)
            X_val = scaler.transform(X_val_raw)
            X_test_scaled = scaler.transform(X_test)

            # -------------------------
            # SMOTE (TRAIN ONLY, 1:2)
            # -------------------------
            smote = SMOTE(
                sampling_strategy=0.5,  # minority : majority = 1 : 2
                random_state=RANDOM_STATE
            )

            X_train, y_train = smote.fit_resample(
                X_train_scaled,
                y_train
            )

            # -------------------------
            # Build Model
            # -------------------------
            model = Sequential()
            initializer = GlorotUniform(seed=RANDOM_STATE)
            input_dim = X_train.shape[1]

            for i, units in enumerate(layers):
                if i == 0:
                    model.add(Dense(
                        units,
                        activation='relu',
                        kernel_initializer=initializer,
                        input_dim=input_dim
                    ))
                else:
                    model.add(Dense(
                        units,
                        activation='relu',
                        kernel_initializer=initializer
                    ))
                model.add(BatchNormalization())
                model.add(Dropout(0.3))

            model.add(Dense(1, activation='sigmoid'))

            model.compile(
                optimizer='adam',
                loss='binary_crossentropy',
                metrics=['accuracy']
            )

            callbacks = [
                EarlyStopping(
                    monitor='val_loss',
                    patience=10,
                    restore_best_weights=True
                ),
                ReduceLROnPlateau(
                    monitor='val_loss',
                    patience=5,
                    factor=0.5
                )
            ]

            model.fit(
                X_train,
                y_train,
                validation_data=(X_val, y_val),
                epochs=EPOCHS,
                batch_size=BATCH_SIZE,
                verbose=0,
                callbacks=callbacks
            )

            # -------------------------
            # Validation Metrics
            # -------------------------
            y_val_prob = model.predict(X_val, verbose=0).ravel()
            y_val_pred = (y_val_prob >= 0.5).astype(int)

            fold_acc.append(
                accuracy_score(y_val, y_val_pred)
            )
            fold_f1.append(
                f1_score(y_val, y_val_pred, average='macro')
            )

        # =========================
        # Store CV Results
        # =========================
        results.extend([
            {
                "Dataset": file,
                "Model": config_name,
                "Split": "Cross-Validation",
                "Metric": "Accuracy",
                "Mean": np.mean(fold_acc),
                "Std": np.std(fold_acc)
            },
            {
                "Dataset": file,
                "Model": config_name,
                "Split": "Cross-Validation",
                "Metric": "Macro-F1",
                "Mean": np.mean(fold_f1),
                "Std": np.std(fold_f1)
            }
        ])

        # =========================
        # Final Test Evaluation
        # =========================
        y_test_prob = model.predict(X_test_scaled, verbose=0).ravel()
        y_test_pred = (y_test_prob >= 0.5).astype(int)

        cm = confusion_matrix(y_test, y_test_pred)

        fpr, tpr, _ = roc_curve(y_test, y_test_prob)
        roc_auc = roc_auc_score(y_test, y_test_prob)

        precision, recall, _ = precision_recall_curve(y_test, y_test_prob)
        pr_auc = auc(recall, precision)

        confusion_matrices.append((file, config_name, cm))
        roc_curves.append((file, config_name, fpr, tpr, roc_auc))
        pr_curves.append((file, config_name, precision, recall, pr_auc))

        results.extend([
            {
                "Dataset": file,
                "Model": config_name,
                "Split": "Test",
                "Metric": "Accuracy",
                "Mean": accuracy_score(y_test, y_test_pred),
                "Std": None
            },
            {
                "Dataset": file,
                "Model": config_name,
                "Split": "Test",
                "Metric": "ROC-AUC",
                "Mean": roc_auc,
                "Std": None
            },
            {
                "Dataset": file,
                "Model": config_name,
                "Split": "Test",
                "Metric": "PR-AUC",
                "Mean": pr_auc,
                "Std": None
            }
        ])

# =========================
# Save Results
# =========================
results_df = pd.DataFrame(results)
results_df.to_csv("Result_CV_BINARY_SIGMOID_SMOTE_1to2.csv", index=False)

# =========================
# Confusion Matrices
# =========================
for dataset, model_name, cm in confusion_matrices:
    plt.figure(figsize=(4, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar=False
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"{dataset} – {model_name}")
    plt.tight_layout()
    plt.show()

# =========================
# ROC Curves
# =========================
plt.figure(figsize=(6, 5))
for dataset, model_name, fpr, tpr, auc_val in roc_curves:
    plt.plot(
        fpr,
        tpr,
        label=f"{dataset}-{model_name} (AUC={auc_val:.3f})"
    )

plt.plot([0, 1], [0, 1], linestyle="--", color="gray")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves (Test Set)")
plt.legend()
plt.tight_layout()
plt.show()

# =========================
# Precision–Recall Curves
# =========================
plt.figure(figsize=(7, 6), dpi=150)

pos_rate = np.mean(y_test)
plt.hlines(
    y=pos_rate,
    xmin=0,
    xmax=1,
    linestyles="dashed",
    linewidth=2,
    label=f"No-skill (pos rate = {pos_rate:.2f})"
)

for dataset, model_name, precision, recall, auc_val in pr_curves:
    plt.plot(
        recall,
        precision,
        linewidth=2,
        label=f"{dataset} | {model_name} (AUC={auc_val:.3f})"
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall Curves (Test Set)")
plt.legend(loc="lower left", fontsize=9)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print("✅ Journal-ready evaluation with SMOTE (1:2) completed.")
